In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from datasets import load_dataset
import wandb
import numpy as np


In [33]:
from datasets import load_dataset

ds = load_dataset("Chiranjeev007/CIFAR-10_Subset")
print(ds)
# DatasetDict({
#   train:      Dataset(num_rows: 5000),
#   validation: Dataset(num_rows: 500),
#   test:       Dataset(num_rows: 1000)
# })

sample = ds["train"][0]
sample["image"]   # PIL Image 32×32 RGB
sample["label"]   # int 0–9

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 500
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})


9

In [34]:
train_ds = ds["train"]
test_ds = ds["test"]
val_ds = ds["validation"]

In [35]:
train_ds, val_ds, test_ds

(Dataset({
     features: ['image', 'label'],
     num_rows: 5000
 }),
 Dataset({
     features: ['image', 'label'],
     num_rows: 500
 }),
 Dataset({
     features: ['image', 'label'],
     num_rows: 1000
 }))


# Transformations

In [36]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [37]:
class HFDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        image = sample["image"]
        label = sample["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [38]:
train_dataset = HFDataset(train_ds, train_transform)
val_dataset   = HFDataset(val_ds, test_transform)
test_dataset  = HFDataset(test_ds, test_transform)

train_loader = DataLoader(train_dataset,
                          batch_size=64,
                          shuffle=True,
                          num_workers=2)

val_loader = DataLoader(val_dataset,
                        batch_size=64,
                        shuffle=False)

test_loader = DataLoader(test_dataset,
                         batch_size=64,
                         shuffle=False)

In [39]:
wandb.init(
    project="cifar10-resnet18",
    config={
        "epochs": 20,
        "batch_size": 64,
        "lr": 2e-4
    }
)

config = wandb.config

epoch,▁▃▆█
train_accuracy,▁▅▇█
train_loss,█▄▂▁
val_accuracy,▇▆█▁
val_loss,▁▅▁█
epoch,3
train_accuracy,0.96183
train_loss,0.11861
val_accuracy,0.85427
val_loss,0.4191


In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights="IMAGENET1K_V1")

# change classifier
num_classes = 10
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

In [41]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=config.lr
)

In [42]:
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)
    return (preds == labels).sum().item() / labels.size(0)

In [43]:
def train_one_epoch(loader):
    model.train()

    total_loss = 0
    total_acc = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(outputs, labels)

    return total_loss/len(loader), total_acc/len(loader)

In [44]:
@torch.no_grad()
def evaluate(loader):
    model.eval()

    total_loss = 0
    total_acc = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        total_acc += accuracy(outputs, labels)

    return total_loss/len(loader), total_acc/len(loader)

In [45]:
best_val_acc = 0
save_path = "best_model.pt"

for epoch in range(config.epochs):

    train_loss, train_acc = train_one_epoch(train_loader)
    val_loss, val_acc = evaluate(val_loader)

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "model_state_dict": model.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, save_path)

        print("✅ Best model saved!")

    print(f"""
    Epoch {epoch+1}
    Train Loss: {train_loss:.4f}
    Train Acc : {train_acc:.4f}
    Val Loss  : {val_loss:.4f}
    Val Acc   : {val_acc:.4f}
    """)

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

✅ Best model saved!

    Epoch 1
    Train Loss: 0.7904
    Train Acc : 0.7352
    Val Loss  : 0.5415
    Val Acc   : 0.8295
    
✅ Best model saved!

    Epoch 2
    Train Loss: 0.3348
    Train Acc : 0.8946
    Val Loss  : 0.4017
    Val Acc   : 0.8669
    

    Epoch 3
    Train Loss: 0.2117
    Train Acc : 0.9343
    Val Loss  : 0.4896
    Val Acc   : 0.8508
    

    Epoch 4
    Train Loss: 0.1850
    Train Acc : 0.9383
    Val Loss  : 0.4193
    Val Acc   : 0.8655
    
✅ Best model saved!

    Epoch 5
    Train Loss: 0.1117
    Train Acc : 0.9658
    Val Loss  : 0.3413
    Val Acc   : 0.8948
    

    Epoch 6
    Train Loss: 0.0973
    Train Acc : 0.9717
    Val Loss  : 0.3331
    Val Acc   : 0.8855
    
✅ Best model saved!

    Epoch 7
    Train Loss: 0.1069
    Train Acc : 0.9686
    Val Loss  : 0.3275
    Val Acc   : 0.9020
    

    Epoch 8
    Train Loss: 0.0557
    Train Acc : 0.9842
    Val Loss  : 0.3516
    Val Acc   : 0.8962
    

    Epoch 9
    Train Loss: 0.0465
    

In [46]:
test_loss, test_acc = evaluate(test_loader)

wandb.log({
    "test_loss": test_loss,
    "test_accuracy": test_acc
})

print("Test Accuracy:", test_acc)

Test Accuracy: 0.9029296875


In [47]:
checkpoint = torch.load("best_model.pt")

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
from huggingface_hub import create_repo, upload_folder, hf_hub_download, login, HfApi

login(token = "")
api = HfApi()

repo_id = "themaverick1/cifar10-resnet18-mldlops-minor"
create_repo(repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj="best_model.pt",   # local file
    path_in_repo="best_model.pt",      # name on HF repo
    repo_id=repo_id
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  best_model.pt               :  75%|#######4  | 33.5MB / 44.8MB            

CommitInfo(commit_url='https://huggingface.co/themaverick1/cifar10-resnet18-mldlops-minor/commit/05b7f642c5247f4b64591f5f149e067d0df357a3', commit_message='Upload best_model.pt with huggingface_hub', commit_description='', oid='05b7f642c5247f4b64591f5f149e067d0df357a3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/themaverick1/cifar10-resnet18-mldlops-minor', endpoint='https://huggingface.co', repo_type='model', repo_id='themaverick1/cifar10-resnet18-mldlops-minor'), pr_revision=None, pr_num=None)

In [53]:
path = hf_hub_download(
    repo_id=repo_id,
    filename="best_model.pt"
)

checkpoint = torch.load(path)

best_model.pt:   0%|          | 0.00/44.8M [00:00<?, ?B/s]

In [54]:
config = {
    "model_type": "resnet18",
    "num_classes": 10,
    "image_size": 224
}


In [55]:
model_new = models.resnet18()
model_new.fc = torch.nn.Linear(model_new.fc.in_features, 10)

model_new.load_state_dict(checkpoint["model_state_dict"])
model_new.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [56]:
class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

In [58]:
all_preds = []
all_labels = []
all_images = []

model_new = model_new.to(device)
model_new.eval()

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_new(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_images.extend(images.cpu())

In [59]:
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [60]:
wandb.log({
    "confusion_matrix":
    wandb.plot.confusion_matrix(
        preds=all_preds,
        y_true=all_labels,
        class_names=class_names
    )
})

In [61]:
class_correct = np.zeros(len(class_names))
class_total = np.zeros(len(class_names))

for pred, label in zip(all_preds, all_labels):
    class_total[label] += 1
    if pred == label:
        class_correct[label] += 1

class_accuracy = class_correct / class_total

In [62]:
import pandas as pd

acc_df = pd.DataFrame({
    "class": class_names,
    "accuracy": class_accuracy
})

wandb.log({
    "class_accuracy":
    wandb.plot.bar(
        wandb.Table(dataframe=acc_df),
        "class",
        "accuracy",
        title="Class-wise Test Accuracy"
    )
})

In [63]:
mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

def denormalize(img):
    return img * std + mean

In [64]:
correct_samples = []
incorrect_samples = []

for img, pred, label in zip(all_images, all_preds, all_labels):

    if pred == label and len(correct_samples) < 10:
        correct_samples.append((img, pred, label))

    elif pred != label and len(incorrect_samples) < 10:
        incorrect_samples.append((img, pred, label))

    if len(correct_samples) == 10 and len(incorrect_samples) == 10:
        break

In [65]:
wandb_images = []

samples = correct_samples + incorrect_samples

for img, pred, label in samples:

    img = denormalize(img).clamp(0,1)

    caption = (
        f"Pred: {class_names[pred]} | "
        f"Actual: {class_names[label]}"
    )

    wandb_images.append(
        wandb.Image(
            img.permute(1,2,0).numpy(),
            caption=caption
        )
    )

wandb.log({
    "Test Predictions (Correct + Incorrect)": wandb_images
})

In [67]:
!git clone https://github.com/themaverick/MLOps-B22CH045-Minor.git

Cloning into 'MLOps-B22CH045-Minor'...


In [68]:
%cd MLOps-B22CH045-Minor

/content/MLOps-B22CH045-Minor


In [69]:
import os
os.makedirs("results", exist_ok=True)

In [72]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.savefig("results/confusion_matrix.png",
            bbox_inches="tight")
plt.close()

In [73]:
plt.figure(figsize=(10,6))

plt.bar(class_names, class_accuracy)
plt.xticks(rotation=45)
plt.ylabel("Accuracy")
plt.title("Class-wise Test Accuracy")

plt.savefig("results/class_accuracy.png",
            bbox_inches="tight")
plt.close()

In [74]:
samples = correct_samples + incorrect_samples

fig, axes = plt.subplots(4,5, figsize=(15,10))
axes = axes.flatten()

for ax, (img, pred, label) in zip(axes, samples):

    img = denormalize(img).clamp(0,1)
    img = img.permute(1,2,0).numpy()

    ax.imshow(img)

    color = "green" if pred==label else "red"

    ax.set_title(
        f"P:{class_names[pred]}\nGT:{class_names[label]}",
        color=color,
        fontsize=9
    )

    ax.axis("off")

plt.tight_layout()
plt.savefig("results/test_predictions.png")
plt.close()

In [75]:
import json

metrics = {
    "test_accuracy": float(test_acc)
}

with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [76]:
!git config --global user.email "sharmayogesh7975@gmail.com"
!git config --global user.name "themaverick"

In [77]:
!git add results/

In [78]:
!git commit -m "Add evaluation results: confusion matrix, class accuracy and predictions"

[main (root-commit) 40b0d1c] Add evaluation results: confusion matrix, class accuracy and predictions
 4 files changed, 3 insertions(+)
 create mode 100644 results/class_accuracy.png
 create mode 100644 results/confusion_matrix.png
 create mode 100644 results/metrics.json
 create mode 100644 results/test_predictions.png


In [79]:
!git push

fatal: could not read Username for 'https://github.com': No such device or address
